In [1]:
import pandas as pd, numpy as np
from pathlib import Path
OUT = Path("dashboard_data"); OUT.mkdir(exist_ok=True)

orders    = pd.read_csv("orders_clean.csv", parse_dates=["order_date"])
customers = pd.read_csv("customers_clean_v2.csv", parse_dates=["registration_date"])
items     = pd.read_csv("order_items_clean.csv")
returns   = pd.read_csv("returns_clean.csv", parse_dates=["return_date"])
inventory = pd.read_csv("inventory_clean.csv")
products  = pd.read_csv("products_clean.csv")

orders_dash = orders.merge(customers[["customer_id", "segment"]], on="customer_id", how="left")
orders_dash["order_month"] = orders_dash["order_date"].dt.strftime("%Y-%m")
orders_dash = orders_dash[["order_id", "customer_id", "order_date", "order_month", "status", "state", "city",
                           "payment_method", "channel", "is_cod", "total_amount", "final_amount", "segment"]]

items_dash = (items.merge(orders[["order_id", "order_date", "status"]].rename(columns={"status": "order_status"}), on="order_id")
                   .merge(products[["product_id", "brand"]], on="product_id", how="left"))
items_dash["order_has_return"] = items_dash["order_id"].isin(returns["order_id"]).astype(int)
items_dash = items_dash[["item_id", "order_id", "product_id", "product_name", "category", "brand", "quantity",
                         "total_price", "order_date", "order_status", "order_has_return"]]

returns_dash = returns.merge(orders[["order_id", "order_date", "status", "state"]].rename(columns={"status": "order_status"}), on="order_id")
returns_dash = returns_dash[["return_id", "order_id", "customer_id", "return_date", "order_date", "reason", "return_amount",
                             "refund_status", "return_condition", "order_status", "state"]]

customers_dash = customers[["customer_id", "state", "city", "gender", "age", "segment", "registration_date", "total_orders", "total_spent"]].copy()
customers_dash["registered_in_window"] = customers_dash["registration_date"].between("2023-06-24", "2025-06-23").astype(int)

inventory_dash = inventory.merge(products[["product_id", "product_name", "category", "brand"]], on="product_id", how="left")
inventory_dash["stock_alert"] = (inventory_dash["status"] != "In Stock").astype(int)
inventory_dash["units_below_reorder"] = (inventory_dash["reorder_level"] - inventory_dash["quantity_available"]).clip(lower=0)

for name, df in {"orders_dash": orders_dash, "items_dash": items_dash, "returns_dash": returns_dash,
                 "customers_dash": customers_dash, "inventory_dash": inventory_dash}.items():
    for col in df.select_dtypes("datetime").columns:
        df[col] = df[col].dt.strftime("%Y-%m-%d")
    df.to_csv(OUT / f"{name}.csv", index=False)
    print(name, df.shape)

orders_dash (50000, 13)
items_dash (100000, 11)
returns_dash (10000, 11)
customers_dash (10000, 10)
inventory_dash (1000, 17)
